# Detekcja naczyń krwionośnych w dnie oka


In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from image_processing import preprocess_image, segment_vessels, get_overlay
from evaluation import calculate_metrics

In [3]:
# Ścieżki do danych
IMAGES_DIR = 'data/images'
MANUAL_DIR = 'data/manual'
MASK_DIR = 'data/mask'

def get_image_list():
    return sorted([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

image_list = get_image_list()

In [ ]:
# Widgety GUI
image_select = widgets.Dropdown(
    options=image_list,
    description='Zdjęcie:',
    disabled=False,
)

method_select = widgets.Dropdown(
    options=['Filtr Frangi', 'Filtr Sato'],
    value='Filtr Frangi',
    description='Metoda:',
    disabled=False,
)

run_button = widgets.Button(
    description='Uruchom',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Uruchom segmentację',
    icon='play'
)

batch_button = widgets.Button(
    description='Statystyki dla 5 zdjęć',
    button_style='info',
    tooltip='Uruchom pełną analizę dla pierwszych 5 zdjęć',
    icon='list'
)

output = widgets.Output()

def display_results(img_name, method):
    img_path = os.path.join(IMAGES_DIR, img_name)
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    base_name = os.path.splitext(img_name)[0]
    manual_path = os.path.join(MANUAL_DIR, base_name + ".tif")
    manual_mask = cv2.imread(manual_path, cv2.IMREAD_GRAYSCALE)
    
    mask_path = os.path.join(MASK_DIR, base_name + "_mask.tif")
    fov_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    preprocessed = preprocess_image(image_rgb)
    detected_vessels = segment_vessels(preprocessed, method=method, mask=fov_mask)
    overlay = get_overlay(image_rgb, detected_vessels)
    
    metrics = calculate_metrics(manual_mask, detected_vessels, mask=fov_mask)
    
    plt.figure(figsize=(20, 10))
    plt.suptitle(f"Wyniki dla: {img_name} (Metoda: {method})", fontsize=16)
    
    plt.subplot(2, 3, 1)
    plt.imshow(image_rgb)
    plt.title('Oryginał')
    plt.axis('off')
    
    plt.subplot(2, 3, 2)
    plt.imshow(detected_vessels, cmap='gray')
    plt.title('Znalezione naczynia')
    plt.axis('off')
    
    plt.subplot(2, 3, 3)
    plt.imshow(manual_mask, cmap='gray')
    plt.title('Maska (Ground Truth)')
    plt.axis('off')
    
    plt.subplot(2, 3, 4)
    plt.imshow(overlay)
    plt.title('Nałożone naczynia')
    plt.axis('off')
    
    plt.subplot(2, 3, 5)
    cm = metrics['confusion_matrix']
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f"Macierz pomyłek\nAcc: {metrics['accuracy']:.4f}, Sens: {metrics['sensitivity']:.4f}\nSpec: {metrics['specificity']:.4f}, G-Mean: {metrics['g_mean']:.4f}")
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ['Background', 'Vessels'])
    plt.yticks(tick_marks, ['Background', 'Vessels'])
    
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()
    return metrics

def on_run_button_clicked(b):
    with output:
        clear_output()
        display_results(image_select.value, method_select.value)

def on_batch_button_clicked(b):
    with output:
        clear_output()
        method = method_select.value
        results = []
        selected_images = image_list[:5]
        
        for img_name in selected_images:
            metrics = display_results(img_name, method)
            results.append({
                'image': img_name,
                'accuracy': metrics['accuracy'],
                'sensitivity': metrics['sensitivity'],
                'specificity': metrics['specificity'],
                'g_mean': metrics['g_mean']
            })
        
        print(f"\n{'PODSUMOWANIE (Metoda: ' + method + ')':^60}")
        print(f"{'Zdjęcie':<15} | {'Acc':<8} | {'Sens':<8} | {'Spec':<8} | {'G-Mean':<8}")
        print("-"*60)
        for r in results:
            print(f"{r['image']:<15} | {r['accuracy']:<8.4f} | {r['sensitivity']:<8.4f} | {r['specificity']:<8.4f} | {r['g_mean']:<8.4f}")

run_button.on_click(on_run_button_clicked)
batch_button.on_click(on_batch_button_clicked)

display(widgets.VBox([image_select, method_select, widgets.HBox([run_button, batch_button]), output]))

### Statystyki dla 5 przykładowych zdjęć
Zgodnie z wymaganiami na ocenę 3.0, poniżej znajduje się podsumowanie statystyczne dla 5 obrazów.

In [ ]:
def run_batch_evaluation(num_images=5):
    results = []
    selected_images = image_list[:num_images]
    
    for img_name in selected_images:
        img_path = os.path.join(IMAGES_DIR, img_name)
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        base_name = os.path.splitext(img_name)[0]
        manual_mask = cv2.imread(os.path.join(MANUAL_DIR, base_name + ".tif"), cv2.IMREAD_GRAYSCALE)
        fov_mask = cv2.imread(os.path.join(MASK_DIR, base_name + "_mask.tif"), cv2.IMREAD_GRAYSCALE)
        
        preprocessed = preprocess_image(image_rgb)
        detected = segment_vessels(preprocessed, mask=fov_mask, method='Filtr Sato')
        metrics = calculate_metrics(manual_mask, detected, mask=fov_mask)
        
        results.append({
            'image': img_name,
            'accuracy': metrics['accuracy'],
            'sensitivity': metrics['sensitivity'],
            'specificity': metrics['specificity'],
            'g_mean': metrics['g_mean']
        })
    
    return results

batch_results = run_batch_evaluation(5)

print(f"{'Zdjęcie':<15} | {'Acc':<8} | {'Sens':<8} | {'Spec':<8} | {'G-Mean':<8}")
print("-"*60)
for r in batch_results:
    print(f"{r['image']:<15} | {r['accuracy']:<8.4f} | {r['sensitivity']:<8.4f} | {r['specificity']:<8.4f} | {r['g_mean']:<8.4f}")

c:\Users\krzys\OneDrive\Pulpit\Informatyka-w-medycynie\Informatyka-w-medycynie\dno_oka\image_processing.py:57: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  cleaned = remove_small_objects(binary_bool, min_size=min_obj_size)
c:\Users\krzys\OneDrive\Pulpit\Informatyka-w-medycynie\Informatyka-w-medycynie\dno_oka\image_processing.py:57: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** it

Zdjęcie         | Acc      | Sens     | Spec     | G-Mean  
------------------------------------------------------------
01_dr.JPG       | 0.9483   | 0.6601   | 0.9668   | 0.7988  
01_g.jpg        | 0.9519   | 0.6285   | 0.9797   | 0.7847  
01_h.jpg        | 0.9386   | 0.5410   | 0.9932   | 0.7330  
02_dr.JPG       | 0.9435   | 0.6290   | 0.9692   | 0.7808  
02_g.jpg        | 0.9493   | 0.6031   | 0.9827   | 0.7699  
